# BỘ KIỂM THỬ TẢI PHASE 1 TRÊN KAGGLE CPU (5 KỊCH BẢN CHUYÊN SÂU)
Hệ thống kiểm thử tải đo lường toàn diện các khía cạnh kiến trúc:
1. **Kịch bản 1: Baseline Read Capacity (50 VUs)** — Đo thông lượng đọc cơ bản.
2. **Kịch bản 2: Stress Testing (100 -> 300 VUs)** — Ép tải tìm điểm gãy (Breaking point).
3. **Kịch bản 3: Auth & Write Lifecycle (30 VUs)** — Đo chi phí CPU của BCrypt và tầng Bảo mật JWT.
4. **Kịch bản 4: Spike Test (Flash-crowd 1.000 VUs)** — Xung lực đột biến tức thì và đo thời gian tự phục hồi (Self-Healing).
5. **Kịch bản 5: High-Throughput Capacity (1.000 -> 2.000 RPS)** — Ép tải theo mô hình Arrival-rate chuẩn công nghiệp.

### Bước 1: Cài đặt OpenJDK 21, MySQL Server và k6 (Non-interactive)

In [ ]:
%%bash
export DEBIAN_FRONTEND=noninteractive
export NEEDRESTART_MODE=a

# Tối ưu hóa giới hạn Socket & File descriptors hệ điều hành cho bài test tải cao (1k - 2k VUs)
ulimit -n 65535
sysctl -w net.ipv4.tcp_tw_reuse=1 2>/dev/null || true
sysctl -w net.core.somaxconn=4096 2>/dev/null || true

apt-get update -qq
apt-get install -y -qq --no-install-recommends openjdk-21-jre-headless mysql-server curl

if [ ! -f /usr/local/bin/k6 ]; then
    curl -s https://github.com/grafana/k6/releases/download/v0.49.0/k6-v0.49.0-linux-amd64.tar.gz -L | tar xvz
    mv k6-v0.49.0-linux-amd64/k6 /usr/local/bin/
    rm -rf k6-v0.49.0-linux-amd64
fi

java -version
k6 version
echo "==> BƯỚC 1 HOÀN TẤT THÀNH CÔNG (Đã nâng cấp OS limit: ulimit -n 65535)!"

### Bước 2: Khởi động MySQL và Khởi tạo Database

In [ ]:
%%bash
mkdir -p /var/run/mysqld
chown -R mysql:mysql /var/run/mysqld /var/lib/mysql
service mysql start

for i in {1..15}; do
    if mysqladmin ping --silent; then
        echo "==> MySQL daemon đã sẵn sàng!"
        break
    fi
    sleep 1
done

mysql -u root -e "CREATE DATABASE IF NOT EXISTS job_application_db;"
mysql -u root -e "CREATE USER IF NOT EXISTS 'job_db_user'@'localhost' IDENTIFIED BY 'job_db_password';"
mysql -u root -e "GRANT ALL PRIVILEGES ON job_application_db.* TO 'job_db_user'@'localhost';"
mysql -u root -e "FLUSH PRIVILEGES;"

echo "==> BƯỚC 2 HOÀN TẤT: Database job_application_db đã sẵn sàng!"

### Bước 3: Khởi chạy Spring Boot dưới nền (Python Subprocess)

In [ ]:
import subprocess, glob

jar_files = glob.glob('/kaggle/input/**/app.jar', recursive=True) + \
            glob.glob('/kaggle/input/**/*.jar', recursive=True) + \
            glob.glob('/kaggle/working/**/app.jar', recursive=True)
jar_files = [f for f in jar_files if not f.endswith('.original')]

if not jar_files:
    raise FileNotFoundError("CHƯA TÌM THẤY file app.jar! Hãy đảm bảo bạn đã Add Dataset chứa file JAR vào Notebook.")

jar_path = jar_files[0]
print(f"==> Đang dùng file: {jar_path}")

log_file = open("backend.log", "w")
# Kích hoạt Java 21 Virtual Threads + Tắt SQL Console Log + Mở rộng HikariCP cho tải cao 1k - 2k VUs
java_cmd = [
    "java",
    "-Dspring.threads.virtual.enabled=true",
    "-Dspring.jpa.show-sql=false",
    "-Dspring.datasource.hikari.maximum-pool-size=40",
    "-Dserver.tomcat.accept-count=1000",
    "-Xms1g",
    "-Xmx4g",
    "-jar",
    jar_path
]

process = subprocess.Popen(
    java_cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)
print(f"==> BƯỚC 3 HOÀN TẤT: Backend đã chạy dưới nền với Java 21 Virtual Threads! PID: {process.pid}")

### Bước 4: Kiểm tra trạng thái Backend và Nạp Seed Data

In [ ]:
%%bash
echo "==> Chờ 15s cho Spring Boot khởi động và Flyway chạy migration..."
sleep 15

echo "--- 15 dòng log mới nhất của Backend ---"
tail -n 15 backend.log

SEED_PATH=$(find /kaggle/input /kaggle/working -name "seed_data.sql" | head -n 1)
if [ -n "$SEED_PATH" ]; then
    echo "==> Đang nạp seed data từ: $SEED_PATH"
    mysql -u job_db_user -pjob_db_password job_application_db < "$SEED_PATH"
    echo "==> Đã nạp xong seed data!"
fi

echo "==> Gọi thử API GET /api/jobs:"
curl -s --connect-timeout 5 http://localhost:8080/api/jobs?page=0&size=2
echo ""

---
## 🧪 KỊCH BẢN 1: BASELINE READ CAPACITY (50 VUs - 40s)
Đo thông lượng đọc cơ bản của hệ thống Pha 1 (Xem danh sách, tìm kiếm, xem chi tiết).

In [ ]:
%%writefile /kaggle/working/test_scenario1_baseline_read.js
import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  stages: [
    { duration: '10s', target: 30 },
    { duration: '20s', target: 50 },
    { duration: '10s', target: 0 },
  ],
  thresholds: {
    http_req_failed: ['rate<0.05'],
    http_req_duration: ['p(95)<1500'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  const resList = http.get(`${BASE_URL}/jobs?page=0&size=10`);
  check(resList, { '1. GET /jobs 200': (r) => r.status === 200 });
  sleep(0.1);

  const resFilter = http.get(`${BASE_URL}/jobs?keyword=Java&category=IT`);
  check(resFilter, { '2. GET /jobs search 200': (r) => r.status === 200 });
  sleep(0.1);

  const randomJobId = Math.floor(Math.random() * 50) + 1;
  const resDetail = http.get(`${BASE_URL}/jobs/${randomJobId}`);
  check(resDetail, { '3. GET /jobs/{id} 200': (r) => r.status === 200 });
  sleep(0.2);
}

In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 1: BASELINE READ (40s)..."
k6 run --summary-export=summary_scenario1.json /kaggle/working/test_scenario1_baseline_read.js

---
## 🧪 KỊCH BẢN 2: STRESS TEST - TÌM ĐIỂM GÃY (100 -> 300 VUs - 60s)
Ép tải cực hạn để quan sát thời điểm CPU 100%, MySQL cạn Connection Pool và xuất hiện lỗi.

In [ ]:
%%writefile /kaggle/working/test_scenario2_stress_test.js
import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  stages: [
    { duration: '10s', target: 100 },
    { duration: '20s', target: 200 },
    { duration: '20s', target: 300 },
    { duration: '10s', target: 0 },
  ],
  thresholds: {
    http_req_duration: ['p(95)<3000'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  const resList = http.get(`${BASE_URL}/jobs?page=0&size=20`);
  check(resList, { 'GET /jobs 200': (r) => r.status === 200 });

  const resFilter = http.get(`${BASE_URL}/jobs?keyword=Engineer&location=Hanoi`);
  check(resFilter, { 'GET /jobs search 200': (r) => r.status === 200 });

  const randomJobId = Math.floor(Math.random() * 50) + 1;
  const resDetail = http.get(`${BASE_URL}/jobs/${randomJobId}`);
  check(resDetail, { 'GET /jobs/{id} 200': (r) => r.status === 200 });

  sleep(0.05);
}

In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 2: STRESS TEST (60s)..."
k6 run --summary-export=summary_scenario2.json /kaggle/working/test_scenario2_stress_test.js

---
## 🧪 KỊCH BẢN 3: AUTHENTICATION & WRITE OPERATIONS (30 VUs - 40s)
Đo đạc chi phí mã hóa BCrypt của CPU và khả năng xử lý nghiệp vụ Ghi + Xác thực JWT.

In [ ]:
%%writefile /kaggle/working/test_scenario3_auth_and_write.js
import http from 'k6/http';
import { check, sleep } from 'k6';

export const options = {
  stages: [
    { duration: '10s', target: 15 },
    { duration: '20s', target: 30 },
    { duration: '10s', target: 0 },
  ],
  thresholds: {
    http_req_failed: ['rate<0.05'],
    http_req_duration: ['p(95)<2500'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  const uniqueId = `${__VU}_${__ITER}_${Date.now()}`;
  const userEmail = `candidate_${uniqueId}@test.com`;
  const userPassword = 'Password123';

  // 1. Đăng ký tài khoản ứng viên (Write + BCrypt Hash)
  const registerPayload = JSON.stringify({
    email: userEmail,
    password: userPassword,
    fullName: `Candidate ${uniqueId}`,
    phone: '0912345678',
    location: 'Hanoi'
  });
  const headers = { 'Content-Type': 'application/json' };
  const regRes = http.post(`${BASE_URL}/auth/applicants/register`, registerPayload, { headers });
  check(regRes, { '1. POST /register 201': (r) => r.status === 201 });
  sleep(0.1);

  // 2. Đăng nhập lấy JWT (BCrypt Verify + JWT Sign)
  const loginPayload = JSON.stringify({
    email: userEmail,
    password: userPassword,
  });
  const loginRes = http.post(`${BASE_URL}/auth/applicants/login`, loginPayload, { headers });
  const loginSuccess = check(loginRes, { '2. POST /login 200': (r) => r.status === 200 });

  // 3. Đọc có bảo mật: Lấy profile bằng Bearer Token (JWT Verification Filter)
  if (loginSuccess && loginRes.json('accessToken')) {
    const token = loginRes.json('accessToken');
    const authHeaders = {
      'Content-Type': 'application/json',
      'Authorization': `Bearer ${token}`
    };
    const profileRes = http.get(`${BASE_URL}/applicants/me`, { headers: authHeaders });
    check(profileRes, { '3. GET /applicants/me 200': (r) => r.status === 200 });
  }
  sleep(0.2);
}

In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 3: AUTH & WRITE (40s)..."
k6 run --summary-export=summary_scenario3.json /kaggle/working/test_scenario3_auth_and_write.js

---
## 🧪 KỊCH BẢN 4: SPIKE TEST - ĐỘT BIẾN TỨC THÌ (1.000 VUs - 65s)
Mô phỏng đợt bùng nổ người dùng tức thì (Flash-crowd: 50 -> 1.000 -> 50 VUs). Đo đạc khả năng chịu xung chấn và tốc độ Tự phục hồi (Self-Healing) của hệ thống.

In [ ]:
%%writefile /kaggle/working/test_scenario4_spike_1000vus.js
import http from 'k6/http';
import { check, sleep } from 'k6';

// =======================================================================
// KỊCH BẢN 4: SPIKE TEST - ĐỘT BIẾN TỨC THÌ 1.000 USERS & TỰ PHỤC HỒI
// Mục tiêu:
// 1. Mô phỏng đợt truy cập ồ ạt (Flash-crowd / Ngày hội tuyển dụng lớn):
//    Đang từ 50 VUs bình thường, đột ngột nhảy vọt lên 1.000 VUs chỉ trong 10s.
// 2. Kiểm tra xem hệ thống có bị Crash/OOM hay không khi chịu xung lực cực lớn.
// 3. Quan sát khả năng Tự phục hồi (Self-Healing / Recovery): Sau khi hạ tải
//    về 50 VUs, hệ thống mất bao lâu để đưa độ trễ (latency) về mức ban đầu.
// =======================================================================

export const options = {
  stages: [
    { duration: '10s', target: 50 },    // Nấc 1: Tải bình thường (50 users)
    { duration: '10s', target: 1000 },  // Nấc 2: BÙNG NỔ TỨC THÌ (SPIKE) lên 1.000 users
    { duration: '15s', target: 1000 },  // Nấc 3: Duy trì đỉnh sóng xung kích 1.000 users
    { duration: '10s', target: 50 },    // Nấc 4: Hạ nhanh về 50 users (Kiểm tra phục hồi)
    { duration: '15s', target: 50 },    // Nấc 5: Đo đạc SLA sau phục hồi
    { duration: '5s', target: 0 },      // Hạ tải hoàn toàn
  ],
  thresholds: {
    // Trong bài Spike Test, chấp nhận tỷ lệ lỗi tạm thời < 15% tại đỉnh xung lực
    http_req_failed: ['rate<0.15'],
    // 95% request phải phản hồi dưới 3s
    http_req_duration: ['p(95)<3000'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  // 1. Lấy danh sách việc làm
  const resList = http.get(`${BASE_URL}/jobs?page=0&size=10`);
  check(resList, {
    '1. GET /jobs 200': (r) => r.status === 200,
  });

  // Nghỉ ngắn giữa các thao tác (Think time mô phỏng người dùng thật)
  sleep(0.3);

  // 2. Tìm kiếm việc làm theo từ khóa
  const resSearch = http.get(`${BASE_URL}/jobs?keyword=Java&category=IT`);
  check(resSearch, {
    '2. GET /jobs search 200': (r) => r.status === 200,
  });

  sleep(0.3);

  // 3. Xem chi tiết ngẫu nhiên một công việc (ID từ 1 đến 50)
  const randomJobId = Math.floor(Math.random() * 50) + 1;
  const resDetail = http.get(`${BASE_URL}/jobs/${randomJobId}`);
  check(resDetail, {
    '3. GET /jobs/{id} 200': (r) => r.status === 200,
  });

  sleep(0.4);
}


In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 4: SPIKE TEST 1.000 VUs (65s)..."
k6 run --summary-export=summary_scenario4.json /kaggle/working/test_scenario4_spike_1000vus.js

---
## 🧪 KỊCH BẢN 5: HIGH-THROUGHPUT CAPACITY (ÉP 1.000 -> 2.000 RPS - 65s)
Áp dụng chuẩn công nghiệp Arrival-rate để tìm điểm bão hòa thông lượng (Saturation Point), kiểm tra sức mạnh xử lý đồng thời của Java 21 Virtual Threads.

In [ ]:
%%writefile /kaggle/working/test_scenario5_high_rps_2000.js
import http from 'k6/http';
import { check } from 'k6';

// =======================================================================
// KỊCH BẢN 5: HIGH-THROUGHPUT CAPACITY (ÉP 1.000 -> 2.000 REQUESTS/GIÂY)
// Mục tiêu:
// 1. Áp dụng mô hình chuẩn công nghiệp: Open Workload Model (Arrival Rate).
// 2. Không phụ thuộc vào số VUs tĩnh mà trực tiếp ép mục tiêu Throughput:
//    500 RPS -> 1.000 RPS -> 2.000 RPS.
// 3. Tìm điểm bão hòa thông lượng (Throughput Saturation Point) của tầng
//    Tomcat (Virtual Threads) và MySQL HikariCP Connection Pool.
// =======================================================================

export const options = {
  scenarios: {
    high_throughput_challenge: {
      executor: 'ramping-arrival-rate',
      startRate: 200,            // Bắt đầu từ 200 requests/s
      timeUnit: '1s',
      preAllocatedVUs: 300,      // Khởi tạo sẵn 300 VUs trong bộ nhớ
      maxVUs: 2500,              // Cho phép k6 tự động scale tối đa 2.500 VUs nếu hệ thống chậm
      stages: [
        { target: 500, duration: '15s' },   // Nấc 1: Tăng lên 500 RPS (Khởi động)
        { target: 1000, duration: '20s' },  // Nấc 2: Chinh phục mốc 1.000 RPS
        { target: 2000, duration: '20s' },  // Nấc 3: Ép cực hạn 2.000 RPS
        { target: 0, duration: '10s' },     // Hạ tải về 0
      ],
    },
  },
  thresholds: {
    // Tỷ lệ lỗi cho phép dưới 10% tại tải cực đại 2.000 RPS
    http_req_failed: ['rate<0.10'],
    // 95% request hoàn thành trong vòng 2 giây
    http_req_duration: ['p(95)<2000'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  // Phân bổ ngẫu nhiên các loại request đọc theo tỷ lệ thực tế:
  // 50% Xem danh sách phân trang, 30% Tìm kiếm theo từ khóa/danh mục, 20% Xem chi tiết tin
  const rand = Math.random();

  if (rand < 0.50) {
    // 50%: Lấy danh sách việc làm (trang ngẫu nhiên từ 0 đến 4)
    const page = Math.floor(Math.random() * 5);
    const res = http.get(`${BASE_URL}/jobs?page=${page}&size=10`);
    check(res, { 'GET /jobs 200': (r) => r.status === 200 });
  } else if (rand < 0.80) {
    // 30%: Tìm kiếm việc làm
    const keywords = ['Java', 'Engineer', 'Developer', 'Backend', 'Senior'];
    const kw = keywords[Math.floor(Math.random() * keywords.length)];
    const res = http.get(`${BASE_URL}/jobs?keyword=${kw}&category=IT`);
    check(res, { 'GET /jobs search 200': (r) => r.status === 200 });
  } else {
    // 20%: Xem chi tiết việc làm ngẫu nhiên (ID từ 1 đến 50)
    const jobId = Math.floor(Math.random() * 50) + 1;
    const res = http.get(`${BASE_URL}/jobs/${jobId}`);
    check(res, { 'GET /jobs/{id} 200': (r) => r.status === 200 });
  }
}


In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 5: HIGH-THROUGHPUT CAPACITY 1.000 -> 2.000 RPS (65s)..."
k6 run --summary-export=summary_scenario5.json /kaggle/working/test_scenario5_high_rps_2000.js

---
## 📊 BẢNG TỔNG HỢP VÀ SO SÁNH TẤT CẢ KỊCH BẢN (1 -> 5)
Tự động đọc file JSON kết quả xuất ra bảng đối chiếu trực tiếp trên Kaggle.

In [ ]:
import json, glob, os
import pandas as pd

scenario_names = {
    'summary_scenario1.json': '1. Baseline Read (50 VUs)',
    'summary_scenario2.json': '2. Stress Test (300 VUs)',
    'summary_scenario3.json': '3. Auth & Write (30 VUs)',
    'summary_scenario4.json': '4. Spike Test (1.000 VUs)',
    'summary_scenario5.json': '5. High-Throughput (2.000 RPS)'
}

records = []
for fname, label in scenario_names.items():
    fpath = os.path.join('/kaggle/working', fname)
    if not os.path.exists(fpath):
        continue
    with open(fpath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    metrics = data.get('metrics', {})
    dur = metrics.get('http_req_duration', {}).get('values', {})
    reqs = metrics.get('http_reqs', {}).get('values', {})
    failed = metrics.get('http_req_failed', {}).get('values', {})
    
    records.append({
        'Kịch bản': label,
        'Tổng Requests': reqs.get('count', 0),
        'Throughput (RPS)': round(reqs.get('rate', 0.0), 2),
        'Avg Latency (ms)': round(dur.get('avg', 0.0), 2),
        'P95 Latency (ms)': round(dur.get('p(95)', 0.0), 2),
        'Max Latency (ms)': round(dur.get('max', 0.0), 2),
        'Tỷ lệ lỗi (%)': round(failed.get('rate', 0.0) * 100, 2)
    })

if records:
    df = pd.DataFrame(records)
    print("=" * 95)
    print("                 BẢNG ĐỐI CHIẾU HIỆU NĂNG TẤT CẢ KỊCH BẢN KIỂM THỬ TẢI")
    print("=" * 95)
    display(df)
else:
    print("Chưa tìm thấy file summary JSON nào. Hãy đảm bảo bạn đã chạy xong các kịch bản kiểm thử!")

---
## 🧪 KỊCH BẢN 4: SPIKE TEST - ĐỘT BIẾN TỨC THÌ (1.000 VUs - 65s)
Mô phỏng đợt bùng nổ người dùng tức thì (Flash-crowd: 50 -> 1.000 -> 50 VUs). Đo đạc khả năng chịu xung chấn và tốc độ Tự phục hồi (Self-Healing) của hệ thống.

In [ ]:
%%writefile /kaggle/working/test_scenario4_spike_1000vus.js
import http from 'k6/http';
import { check, sleep } from 'k6';

// =======================================================================
// KỊCH BẢN 4: SPIKE TEST - ĐỘT BIẾN TỨC THÌ 1.000 USERS & TỰ PHỤC HỒI
// Mục tiêu:
// 1. Mô phỏng đợt truy cập ồ ạt (Flash-crowd / Ngày hội tuyển dụng lớn):
//    Đang từ 50 VUs bình thường, đột ngột nhảy vọt lên 1.000 VUs chỉ trong 10s.
// 2. Kiểm tra xem hệ thống có bị Crash/OOM hay không khi chịu xung lực cực lớn.
// 3. Quan sát khả năng Tự phục hồi (Self-Healing / Recovery): Sau khi hạ tải
//    về 50 VUs, hệ thống mất bao lâu để đưa độ trễ (latency) về mức ban đầu.
// =======================================================================

export const options = {
  stages: [
    { duration: '10s', target: 50 },    // Nấc 1: Tải bình thường (50 users)
    { duration: '10s', target: 1000 },  // Nấc 2: BÙNG NỔ TỨC THÌ (SPIKE) lên 1.000 users
    { duration: '15s', target: 1000 },  // Nấc 3: Duy trì đỉnh sóng xung kích 1.000 users
    { duration: '10s', target: 50 },    // Nấc 4: Hạ nhanh về 50 users (Kiểm tra phục hồi)
    { duration: '15s', target: 50 },    // Nấc 5: Đo đạc SLA sau phục hồi
    { duration: '5s', target: 0 },      // Hạ tải hoàn toàn
  ],
  thresholds: {
    // Trong bài Spike Test, chấp nhận tỷ lệ lỗi tạm thời < 15% tại đỉnh xung lực
    http_req_failed: ['rate<0.15'],
    // 95% request phải phản hồi dưới 3s
    http_req_duration: ['p(95)<3000'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  // 1. Lấy danh sách việc làm
  const resList = http.get(`${BASE_URL}/jobs?page=0&size=10`);
  check(resList, {
    '1. GET /jobs 200': (r) => r.status === 200,
  });

  // Nghỉ ngắn giữa các thao tác (Think time mô phỏng người dùng thật)
  sleep(0.3);

  // 2. Tìm kiếm việc làm theo từ khóa
  const resSearch = http.get(`${BASE_URL}/jobs?keyword=Java&category=IT`);
  check(resSearch, {
    '2. GET /jobs search 200': (r) => r.status === 200,
  });

  sleep(0.3);

  // 3. Xem chi tiết ngẫu nhiên một công việc (ID từ 1 đến 50)
  const randomJobId = Math.floor(Math.random() * 50) + 1;
  const resDetail = http.get(`${BASE_URL}/jobs/${randomJobId}`);
  check(resDetail, {
    '3. GET /jobs/{id} 200': (r) => r.status === 200,
  });

  sleep(0.4);
}


In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 4: SPIKE TEST 1.000 VUs (65s)..."
k6 run --summary-export=summary_scenario4.json /kaggle/working/test_scenario4_spike_1000vus.js

---
## 🧪 KỊCH BẢN 5: HIGH-THROUGHPUT CAPACITY (ÉP 1.000 -> 2.000 RPS - 65s)
Áp dụng chuẩn công nghiệp Arrival-rate để tìm điểm bão hòa thông lượng (Saturation Point), kiểm tra sức mạnh xử lý đồng thời của Java 21 Virtual Threads.

In [ ]:
%%writefile /kaggle/working/test_scenario5_high_rps_2000.js
import http from 'k6/http';
import { check } from 'k6';

// =======================================================================
// KỊCH BẢN 5: HIGH-THROUGHPUT CAPACITY (ÉP 1.000 -> 2.000 REQUESTS/GIÂY)
// Mục tiêu:
// 1. Áp dụng mô hình chuẩn công nghiệp: Open Workload Model (Arrival Rate).
// 2. Không phụ thuộc vào số VUs tĩnh mà trực tiếp ép mục tiêu Throughput:
//    500 RPS -> 1.000 RPS -> 2.000 RPS.
// 3. Tìm điểm bão hòa thông lượng (Throughput Saturation Point) của tầng
//    Tomcat (Virtual Threads) và MySQL HikariCP Connection Pool.
// =======================================================================

export const options = {
  scenarios: {
    high_throughput_challenge: {
      executor: 'ramping-arrival-rate',
      startRate: 200,            // Bắt đầu từ 200 requests/s
      timeUnit: '1s',
      preAllocatedVUs: 300,      // Khởi tạo sẵn 300 VUs trong bộ nhớ
      maxVUs: 2500,              // Cho phép k6 tự động scale tối đa 2.500 VUs nếu hệ thống chậm
      stages: [
        { target: 500, duration: '15s' },   // Nấc 1: Tăng lên 500 RPS (Khởi động)
        { target: 1000, duration: '20s' },  // Nấc 2: Chinh phục mốc 1.000 RPS
        { target: 2000, duration: '20s' },  // Nấc 3: Ép cực hạn 2.000 RPS
        { target: 0, duration: '10s' },     // Hạ tải về 0
      ],
    },
  },
  thresholds: {
    // Tỷ lệ lỗi cho phép dưới 10% tại tải cực đại 2.000 RPS
    http_req_failed: ['rate<0.10'],
    // 95% request hoàn thành trong vòng 2 giây
    http_req_duration: ['p(95)<2000'],
  },
};

const BASE_URL = 'http://localhost:8080/api';

export default function () {
  // Phân bổ ngẫu nhiên các loại request đọc theo tỷ lệ thực tế:
  // 50% Xem danh sách phân trang, 30% Tìm kiếm theo từ khóa/danh mục, 20% Xem chi tiết tin
  const rand = Math.random();

  if (rand < 0.50) {
    // 50%: Lấy danh sách việc làm (trang ngẫu nhiên từ 0 đến 4)
    const page = Math.floor(Math.random() * 5);
    const res = http.get(`${BASE_URL}/jobs?page=${page}&size=10`);
    check(res, { 'GET /jobs 200': (r) => r.status === 200 });
  } else if (rand < 0.80) {
    // 30%: Tìm kiếm việc làm
    const keywords = ['Java', 'Engineer', 'Developer', 'Backend', 'Senior'];
    const kw = keywords[Math.floor(Math.random() * keywords.length)];
    const res = http.get(`${BASE_URL}/jobs?keyword=${kw}&category=IT`);
    check(res, { 'GET /jobs search 200': (r) => r.status === 200 });
  } else {
    // 20%: Xem chi tiết việc làm ngẫu nhiên (ID từ 1 đến 50)
    const jobId = Math.floor(Math.random() * 50) + 1;
    const res = http.get(`${BASE_URL}/jobs/${jobId}`);
    check(res, { 'GET /jobs/{id} 200': (r) => r.status === 200 });
  }
}


In [ ]:
%%bash
echo "==> CHẠY KỊCH BẢN 5: HIGH-THROUGHPUT CAPACITY 1.000 -> 2.000 RPS (65s)..."
k6 run --summary-export=summary_scenario5.json /kaggle/working/test_scenario5_high_rps_2000.js

---
## 📊 BẢNG TỔNG HỢP VÀ SO SÁNH TẤT CẢ KỊCH BẢN (1 -> 5)
Tự động đọc file JSON kết quả xuất ra bảng đối chiếu trực tiếp trên Kaggle.

In [ ]:
import json, glob, os
import pandas as pd

scenario_names = {
    'summary_scenario1.json': '1. Baseline Read (50 VUs)',
    'summary_scenario2.json': '2. Stress Test (300 VUs)',
    'summary_scenario3.json': '3. Auth & Write (30 VUs)',
    'summary_scenario4.json': '4. Spike Test (1.000 VUs)',
    'summary_scenario5.json': '5. High-Throughput (2.000 RPS)'
}

records = []
for fname, label in scenario_names.items():
    fpath = os.path.join('/kaggle/working', fname)
    if not os.path.exists(fpath):
        continue
    with open(fpath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    metrics = data.get('metrics', {})
    dur = metrics.get('http_req_duration', {}).get('values', {})
    reqs = metrics.get('http_reqs', {}).get('values', {})
    failed = metrics.get('http_req_failed', {}).get('values', {})
    
    records.append({
        'Kịch bản': label,
        'Tổng Requests': reqs.get('count', 0),
        'Throughput (RPS)': round(reqs.get('rate', 0.0), 2),
        'Avg Latency (ms)': round(dur.get('avg', 0.0), 2),
        'P95 Latency (ms)': round(dur.get('p(95)', 0.0), 2),
        'Max Latency (ms)': round(dur.get('max', 0.0), 2),
        'Tỷ lệ lỗi (%)': round(failed.get('rate', 0.0) * 100, 2)
    })

if records:
    df = pd.DataFrame(records)
    print("=" * 95)
    print("                 BẢNG ĐỐI CHIẾU HIỆU NĂNG TẤT CẢ KỊCH BẢN KIỂM THỬ TẢI")
    print("=" * 95)
    display(df)
else:
    print("Chưa tìm thấy file summary JSON nào. Hãy đảm bảo bạn đã chạy xong các kịch bản kiểm thử!")